In [1]:
import pandas as pd
import numpy as np

In [2]:
air_reserve = pd.read_csv("air_reserve.csv")
air_store_info = pd.read_csv("air_store_info.csv")
air_visit = pd.read_csv("air_visit_data.csv")
date_info = pd.read_csv("date_info.csv")

print("Air reserve:", air_reserve.shape)
print("Store info:", air_store_info.shape)
print("Visit data:", air_visit.shape)
print("Date info:", date_info.shape)

Air reserve: (92378, 4)
Store info: (829, 5)
Visit data: (252108, 3)
Date info: (517, 3)


In [3]:
air_visit.head()

,air_store_id,visit_date,visitors
0,air_ba937bf13d40fb24,2016-01-13,25
1,air_ba937bf13d40fb24,2016-01-14,32
2,air_ba937bf13d40fb24,2016-01-15,29
3,air_ba937bf13d40fb24,2016-01-16,22
4,air_ba937bf13d40fb24,2016-01-18,6


In [4]:
air_visit.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 252108 entries, 0 to 252107
Data columns (total 3 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   air_store_id  252108 non-null  object
 1   visit_date    252108 non-null  object
 2   visitors      252108 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 5.8+ MB


air_visit je glavna tabela grupisana po restoranu i datumu, prikazuje broj gostiju za odredjeni dan po restoranu

Datum pretvaramo u pravi tip, i sa nunique gledamo koliko ima restorana

In [5]:
air_visit['visit_date']= pd.to_datetime(air_visit['visit_date']).dt.normalize()

In [6]:
air_visit['air_store_id'].nunique()

829

In [7]:
air_visit['month'] = air_visit['visit_date'].dt.month

In [8]:
air_reserve.head()

,air_store_id,visit_datetime,reserve_datetime,reserve_visitors
0,air_877f79706adbfb06,2016-01-01 19:00:00,2016-01-01 16:00:00,1
1,air_db4b38ebe7a7ceff,2016-01-01 19:00:00,2016-01-01 19:00:00,3
2,air_db4b38ebe7a7ceff,2016-01-01 19:00:00,2016-01-01 19:00:00,6
3,air_877f79706adbfb06,2016-01-01 20:00:00,2016-01-01 16:00:00,2
4,air_db80363d35f10926,2016-01-01 20:00:00,2016-01-01 01:00:00,5


In [9]:
air_reserve.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92378 entries, 0 to 92377
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   air_store_id      92378 non-null  object
 1   visit_datetime    92378 non-null  object
 2   reserve_datetime  92378 non-null  object
 3   reserve_visitors  92378 non-null  int64 
dtypes: int64(1), object(3)
memory usage: 2.8+ MB


Air_reserve tabela pokazuje datum rezervacije, datum posete i broj rezervisanih mesta za restoran i konkreto vreme  
Datume je potrebno transformisati u tip datetime

In [10]:
air_reserve['reserve_datetime'] = pd.to_datetime(air_reserve['reserve_datetime'])
air_reserve['visit_datetime'] = pd.to_datetime(air_reserve['visit_datetime'])

Ova tabela se koristi kao feature source  
Jedan red treba da se odnosi na jedan restoran za konkretni datum  

Mozemo kreirati atribute koji nam pokazuju koje je doba dana na osnovu sati

In [11]:
air_reserve['visit_hour'] = air_reserve['visit_datetime'].dt.hour

In [12]:
def time_of_day(hour):
    if hour < 11:
        return "morning"
    elif hour < 17:
        return "afternoon"
    else:
        return "evening"

In [13]:
air_reserve['time_of_day'] = air_reserve['visit_hour'].apply(time_of_day)

In [14]:
print(air_reserve['time_of_day'].value_counts(dropna=False))

time_of_day
evening      80906
afternoon     9944
morning       1528
Name: count, dtype: int64


crosstab radi groupby count i pivot u isto vreme, pravi tabelu frekvencija i broji koliko puta se nesto desilo po grupama

In [15]:
air_reserve['reserve_datetime'] = pd.to_datetime(air_reserve['reserve_datetime']).dt.normalize()
air_reserve['visit_datetime'] = pd.to_datetime(air_reserve['visit_datetime']).dt.normalize()

air_reserve.rename(columns={'reserve_datetime' : 'reserve_date', 'visit_datetime':'visit_date'}, inplace = True)

In [16]:
time_of_day_features = pd.crosstab(
    [air_reserve['air_store_id'], air_reserve['visit_date']],
    air_reserve['time_of_day']
).reset_index()

Kreiramo agregacije, atribut koji pokazuje koliko dana ljudi unapred rezervisu, ukupan broj rezervacija, ukupan broj rezervisanih mesta i prosecan broj dana unapred kada su napravljene rezervacije

In [17]:
air_reserve["reserve_lead_days"] = (
pd.to_datetime(air_reserve["visit_date"]) - pd.to_datetime(air_reserve["reserve_date"])
).dt.days

In [18]:
reserve_features = (
air_reserve.groupby(["air_store_id", "visit_date"])
.agg(
reserve_count=("reserve_visitors", "size"),
reserve_visitors_sum=("reserve_visitors", "sum"),
reserve_lead_days_mean=("reserve_lead_days", "mean")
)
.reset_index()
)

In [19]:
reserve_features = reserve_features.merge(
    time_of_day_features,
    on=['air_store_id', 'visit_date'],
    how='left'
)

In [20]:
reserve_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29830 entries, 0 to 29829
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   air_store_id            29830 non-null  object        
 1   visit_date              29830 non-null  datetime64[ns]
 2   reserve_count           29830 non-null  int64         
 3   reserve_visitors_sum    29830 non-null  int64         
 4   reserve_lead_days_mean  29830 non-null  float64       
 5   afternoon               29830 non-null  int64         
 6   evening                 29830 non-null  int64         
 7   morning                 29830 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(5), object(1)
memory usage: 1.8+ MB


In [21]:
reserve_features.head()

,air_store_id,visit_date,reserve_count,reserve_visitors_sum,reserve_lead_days_mean,afternoon,evening,morning
0,air_00a91d42b08b08d9,2016-10-31,1,2,0.0,0,1,0
1,air_00a91d42b08b08d9,2016-12-05,1,9,4.0,0,1,0
2,air_00a91d42b08b08d9,2016-12-14,1,18,6.0,0,1,0
3,air_00a91d42b08b08d9,2016-12-17,1,2,6.0,0,1,0
4,air_00a91d42b08b08d9,2016-12-20,1,4,2.0,0,1,0


Spajamo glavnu tabelu air_visit sa izvedenim atributima

In [22]:
air_visit = air_visit.merge(reserve_features, on=["air_store_id", "visit_date"], how="left")

Neophodno je za agregacije popuniti nedostajuce vrednosti 0

In [23]:
for col in ["reserve_count", "reserve_visitors_sum","morning", "afternoon", "evening"]:
    air_visit[col] = air_visit[col].fillna(0)

Ovde obratiti paznju da ne treba popunjavati 0 nedostajuce vrednosti atributa reserve_lead_days jer bi to znacilo da je bilo rezervacije istog dana kad i posete, a u ovom slucaju NaN znaci da je nije bilo rezervacije  
Mozemo da napravimo jos jedan atribut has_reservation kao indikator, popunimo 0 ali ce model zbog ovog indikatora znati da li je bilo rezervacije ili ne

In [24]:
air_visit['reserve_lead_days_mean'] = air_visit['reserve_lead_days_mean'].fillna(0)
air_visit['has_reservation'] = (air_visit['reserve_count'] > 0).astype(int)

In [25]:
air_visit.shape

(252108, 11)

In [26]:
air_store_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 829 entries, 0 to 828
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   air_store_id    829 non-null    object 
 1   air_genre_name  829 non-null    object 
 2   air_area_name   829 non-null    object 
 3   latitude        829 non-null    float64
 4   longitude       829 non-null    float64
dtypes: float64(2), object(3)
memory usage: 32.5+ KB


In [27]:
air_store_info.head()

,air_store_id,air_genre_name,air_area_name,latitude,longitude
0,air_0f0cdeee6c9bf3d7,Italian/French,Hyōgo-ken Kōbe-shi Kumoidōri,34.695124,135.197853
1,air_7cc17a324ae5c7dc,Italian/French,Hyōgo-ken Kōbe-shi Kumoidōri,34.695124,135.197853
2,air_fee8dcf4d619598e,Italian/French,Hyōgo-ken Kōbe-shi Kumoidōri,34.695124,135.197853
3,air_a17f0778617c76e2,Italian/French,Hyōgo-ken Kōbe-shi Kumoidōri,34.695124,135.197853
4,air_83db5aff8f50478e,Italian/French,Tōkyō-to Minato-ku Shibakōen,35.658068,139.751599


Ovu tabelu koristimo kako bismo po zahtevu zadatka uradili filter po restoranima u Tokiju

In [28]:
air_store_info.loc[
    air_store_info['air_area_name'].str.contains('Tōkyō', na=False),
    'air_area_name'
].nunique()

51

Proveravamo koliko restorana sadrzi u nazivu Tokyo

In [29]:
air_store_info = air_store_info.loc[air_store_info['air_area_name'].str.contains('Tōkyō', na=False)]

In [30]:
air_store_info.head()

,air_store_id,air_genre_name,air_area_name,latitude,longitude
4,air_83db5aff8f50478e,Italian/French,Tōkyō-to Minato-ku Shibakōen,35.658068,139.751599
5,air_99c3eae84130c1cb,Italian/French,Tōkyō-to Minato-ku Shibakōen,35.658068,139.751599
6,air_f183a514cb8ff4fa,Italian/French,Tōkyō-to Minato-ku Shibakōen,35.658068,139.751599
7,air_6b9fa44a9cf504a1,Italian/French,Tōkyō-to Minato-ku Shibakōen,35.658068,139.751599
8,air_0919d54f0c9a24b8,Italian/French,Tōkyō-to Minato-ku Shibakōen,35.658068,139.751599


In [31]:
air_visit = air_visit.merge(air_store_info, on=["air_store_id"], how="inner")

In [32]:
air_visit.shape

(133063, 15)

In [33]:
date_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   calendar_date  517 non-null    object
 1   day_of_week    517 non-null    object
 2   holiday_flg    517 non-null    int64 
dtypes: int64(1), object(2)
memory usage: 12.2+ KB


In [34]:
date_info.head()

,calendar_date,day_of_week,holiday_flg
0,2016-01-01,Friday,1
1,2016-01-02,Saturday,1
2,2016-01-03,Sunday,1
3,2016-01-04,Monday,0
4,2016-01-05,Tuesday,0


In [35]:
date_info['calendar_date'] = pd.to_datetime(date_info['calendar_date']).dt.normalize()

Ova tabela nam daje informaciju o praznicima i danima u nedelji

Konacno spajanje

In [36]:
air_visit = air_visit.merge(
    date_info,
    left_on='visit_date',
    right_on='calendar_date',
    how='left'
)

In [37]:
air_visit.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 133063 entries, 0 to 133062
Data columns (total 18 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   air_store_id            133063 non-null  object        
 1   visit_date              133063 non-null  datetime64[ns]
 2   visitors                133063 non-null  int64         
 3   month                   133063 non-null  int32         
 4   reserve_count           133063 non-null  float64       
 5   reserve_visitors_sum    133063 non-null  float64       
 6   reserve_lead_days_mean  133063 non-null  float64       
 7   afternoon               133063 non-null  float64       
 8   evening                 133063 non-null  float64       
 9   morning                 133063 non-null  float64       
 10  has_reservation         133063 non-null  int64         
 11  air_genre_name          133063 non-null  object        
 12  air_area_name           133063

In [38]:
air_visit.head()

,air_store_id,visit_date,visitors,month,reserve_count,reserve_visitors_sum,reserve_lead_days_mean,afternoon,evening,morning,has_reservation,air_genre_name,air_area_name,latitude,longitude,calendar_date,day_of_week,holiday_flg
0,air_ba937bf13d40fb24,2016-01-13,25,1,0.0,0.0,0.0,0.0,0.0,0.0,0,Dining bar,Tōkyō-to Minato-ku Shibakōen,35.658068,139.751599,2016-01-13,Wednesday,0
1,air_ba937bf13d40fb24,2016-01-14,32,1,0.0,0.0,0.0,0.0,0.0,0.0,0,Dining bar,Tōkyō-to Minato-ku Shibakōen,35.658068,139.751599,2016-01-14,Thursday,0
2,air_ba937bf13d40fb24,2016-01-15,29,1,0.0,0.0,0.0,0.0,0.0,0.0,0,Dining bar,Tōkyō-to Minato-ku Shibakōen,35.658068,139.751599,2016-01-15,Friday,0
3,air_ba937bf13d40fb24,2016-01-16,22,1,0.0,0.0,0.0,0.0,0.0,0.0,0,Dining bar,Tōkyō-to Minato-ku Shibakōen,35.658068,139.751599,2016-01-16,Saturday,0
4,air_ba937bf13d40fb24,2016-01-18,6,1,0.0,0.0,0.0,0.0,0.0,0.0,0,Dining bar,Tōkyō-to Minato-ku Shibakōen,35.658068,139.751599,2016-01-18,Monday,0


In [39]:
air_visit.drop(columns=["calendar_date"], inplace=True)

In [40]:
air_visit['is_weekend'] = (air_visit['day_of_week']).isin(['Friday', 'Saturday', 'Sunday']).astype(int)

In [41]:
air_visit.to_csv("preprocessed_dataset.csv", index=False)

--------

In [42]:
df = pd.read_csv('preprocessed_dataset.csv')

In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 133063 entries, 0 to 133062
Data columns (total 18 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   air_store_id            133063 non-null  object 
 1   visit_date              133063 non-null  object 
 2   visitors                133063 non-null  int64  
 3   month                   133063 non-null  int64  
 4   reserve_count           133063 non-null  float64
 5   reserve_visitors_sum    133063 non-null  float64
 6   reserve_lead_days_mean  133063 non-null  float64
 7   afternoon               133063 non-null  float64
 8   evening                 133063 non-null  float64
 9   morning                 133063 non-null  float64
 10  has_reservation         133063 non-null  int64  
 11  air_genre_name          133063 non-null  object 
 12  air_area_name           133063 non-null  object 
 13  latitude                133063 non-null  float64
 14  longitude           

Zbog vremenskih serija, sortiramo datum

In [44]:
df = df.sort_values("visit_date").reset_index(drop=True)

Podela na test za obucavanje i testiranje po preseku '2017-04-01'

In [45]:
train = df[df["visit_date"] < "2017-04-01"].copy()
test = df[df["visit_date"] >= "2017-04-01"].copy()

Kreiramo izlaznu promenljivu

In [46]:
median_per_restaurant_train = (
    train.groupby("air_store_id")["visitors"]
    .median()
    .reset_index(name="median_visitors")
)

In [47]:
train = train.merge(median_per_restaurant_train, on="air_store_id", how="left")
test = test.merge(median_per_restaurant_train, on="air_store_id", how="left")

In [48]:
train["target"] = (train["visitors"] > train["median_visitors"]).astype(int)
test["target"] = (test["visitors"] > test["median_visitors"]).astype(int)

In [49]:
cols_to_drop = ["target", "visitors", "air_store_id", "visit_date"]

X_train = train.drop(columns=cols_to_drop)
y_train = train["target"]

X_test = test.drop(columns=cols_to_drop)
y_test = test["target"]

In [50]:
X_train.head()

,month,reserve_count,reserve_visitors_sum,reserve_lead_days_mean,afternoon,evening,morning,has_reservation,air_genre_name,air_area_name,latitude,longitude,day_of_week,holiday_flg,is_weekend,median_visitors
0,1,0.0,0.0,0.0,0.0,0.0,0.0,0,Cafe/Sweets,Tōkyō-to Taitō-ku Higashiueno,35.712607,139.779996,Friday,1,1,16.0
1,1,0.0,0.0,0.0,0.0,0.0,0.0,0,Western food,Tōkyō-to Shibuya-ku Shibuya,35.661777,139.704051,Friday,1,1,24.0
2,1,0.0,0.0,0.0,0.0,0.0,0.0,0,Izakaya,Tōkyō-to Nerima-ku Toyotamakita,35.735623,139.651658,Friday,1,1,34.0
3,1,0.0,0.0,0.0,0.0,0.0,0.0,0,Creative cuisine,Tōkyō-to Shinjuku-ku Kabukichō,35.693840,139.703549,Friday,1,1,17.0
4,1,2.0,3.0,0.0,0.0,2.0,0.0,1,Japanese food,Tōkyō-to Minato-ku Shibakōen,35.658068,139.751599,Friday,1,1,12.0


In [51]:
X_train.isna().sum()

month                     0
reserve_count             0
reserve_visitors_sum      0
reserve_lead_days_mean    0
afternoon                 0
evening                   0
morning                   0
has_reservation           0
air_genre_name            0
air_area_name             0
latitude                  0
longitude                 0
day_of_week               0
holiday_flg               0
is_weekend                0
median_visitors           0
dtype: int64

In [52]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 124619 entries, 0 to 124618
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   month                   124619 non-null  int64  
 1   reserve_count           124619 non-null  float64
 2   reserve_visitors_sum    124619 non-null  float64
 3   reserve_lead_days_mean  124619 non-null  float64
 4   afternoon               124619 non-null  float64
 5   evening                 124619 non-null  float64
 6   morning                 124619 non-null  float64
 7   has_reservation         124619 non-null  int64  
 8   air_genre_name          124619 non-null  object 
 9   air_area_name           124619 non-null  object 
 10  latitude                124619 non-null  float64
 11  longitude               124619 non-null  float64
 12  day_of_week             124619 non-null  object 
 13  holiday_flg             124619 non-null  int64  
 14  is_weekend          

### Logisticka regresija

In [53]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.metrics import precision_score, recall_score, f1_score

categorical_features = ["air_genre_name", "air_area_name","day_of_week"]
numeric_features = [col for col in X_train.columns if col not in categorical_features]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

log_reg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42, C=0.1, penalty="l1", solver="liblinear"))
])

log_reg_pipeline.fit(X_train, y_train)

y_pred = log_reg_pipeline.predict(X_test)
y_proba = log_reg_pipeline.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, pos_label=1))
print("Recall:", recall_score(y_test, y_pred, pos_label=1))
print("F1-score:", f1_score(y_test, y_pred, pos_label=1))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

Accuracy: 0.618427285646613
Precision: 0.6101998974884675
Recall: 0.5832925036746692
F1-score: 0.5964428857715431
ROC-AUC: 0.6599894168626154


### Random Forest

In [54]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.metrics import precision_score, recall_score, f1_score

categorical_features = ["air_genre_name", "air_area_name","day_of_week"]
numeric_features = [col for col in X_train.columns if col not in categorical_features]

preprocessor = ColumnTransformer(transformers=[
    ("num", "passthrough", numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)

y_pred = rf_pipeline.predict(X_test)
y_proba = rf_pipeline.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, pos_label=1))
print("Recall:", recall_score(y_test, y_pred, pos_label=1))
print("F1-score:", f1_score(y_test, y_pred, pos_label=1))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

Accuracy: 0.6427048792041686
Precision: 0.6473029045643154
Recall: 0.5732484076433121
F1-score: 0.608029102247629
ROC-AUC: 0.6958037107701114


In [55]:
#!pip install lightgbm

### LightGBM

In [56]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.metrics import precision_score, recall_score, f1_score

categorical_features = ["air_genre_name", "air_area_name","day_of_week"]
numeric_features = [col for col in X_train.columns if col not in categorical_features]

preprocessor = ColumnTransformer(transformers=[
    ("num", "passthrough", numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

lgbm_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier(
        n_estimators=200,
        learning_rate=0.01,
        num_leaves=31,
        random_state=42
    ))
])

lgbm_pipeline.fit(X_train, y_train)

y_pred = lgbm_pipeline.predict(X_test)
y_proba = lgbm_pipeline.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, pos_label=1))
print("Recall:", recall_score(y_test, y_pred, pos_label=1))
print("F1-score:", f1_score(y_test, y_pred, pos_label=1))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

[LightGBM] [Info] Number of positive: 58571, number of negative: 66048
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007439 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 706
[LightGBM] [Info] Number of data points in the train set: 124619, number of used features: 84
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.470001 -> initscore=-0.120142
[LightGBM] [Info] Start training from score -0.120142
Accuracy: 0.6451918522027476
Precision: 0.641185647425897
Recall: 0.6041156295933366
F1-score: 0.6220988900100908
ROC-AUC: 0.6930570597568732


C:\Users\user\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\user\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [57]:
#!pip install xgboost

### XGBoost

In [58]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.metrics import precision_score, recall_score, f1_score

categorical_features = ["air_genre_name", "air_area_name","day_of_week"]
numeric_features = [col for col in X_train.columns if col not in categorical_features]

preprocessor = ColumnTransformer(transformers=[
    ("num", "passthrough", numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=200,
        learning_rate=0.01,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="logloss"
    ))
])

xgb_pipeline.fit(X_train, y_train)

y_pred = xgb_pipeline.predict(X_test)
y_proba = xgb_pipeline.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, pos_label=1))
print("Recall:", recall_score(y_test, y_pred, pos_label=1))
print("F1-score:", f1_score(y_test, y_pred, pos_label=1))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

Accuracy: 0.6422311700615821
Precision: 0.6478127612148231
Recall: 0.5695737383635473
F1-score: 0.6061791161517404
ROC-AUC: 0.6933482308233708


Optimizacija hiperparametara Random Forest modela

In [59]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier

tscv = TimeSeriesSplit(n_splits=5)

param_grid = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [5, 10],
    "model__min_samples_leaf": [5, 10, 20],
    "model__max_leaf_nodes": [20, 50]
}

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=tscv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Najbolji parametri:", grid_search.best_params_)
print("Najbolji CV ROC-AUC:", grid_search.best_score_)

Fitting 5 folds for each of 24 candidates, totalling 120 fits
Najbolji parametri: {'model__max_depth': 10, 'model__max_leaf_nodes': 50, 'model__min_samples_leaf': 5, 'model__n_estimators': 300}
Najbolji CV ROC-AUC: 0.6653866170513882
